<a href="https://colab.research.google.com/github/cintiafelixdata/analise-experiencia-cliente/blob/main/Analise_de_Negativas_Financeiras.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Projeto: Análise de Negativas Financeiras e Experiência do Cliente

**Objetivo:** identificar dificuldades enfrentadas pelos clientes durante o processo de negativa financeira, analisar os dados e propor melhorias para a experiência do cliente.

In [2]:
import pandas as pd

dados = {
    "id_solicitacao": range(1, 16),
    "idade": [22, 25, 34, 41, 28, 52, 37, 24, 46, 33, 29, 55, 39, 26, 48],
    "motivo_negativa": [
        "Renda insuficiente",
        "Motivo não informado",
        "Documentação incompleta",
        "Renda insuficiente",
        "Motivo não informado",
        "Critérios não atendidos",
        "Documentação incompleta",
        "Renda insuficiente",
        "Motivo não informado",
        "Critérios não atendidos",
        "Renda insuficiente",
        "Documentação incompleta",
        "Motivo não informado",
        "Critérios não atendidos",
        "Renda insuficiente"
    ],
    "motivo_informado": [
        "Sim", "Não", "Sim", "Sim", "Não",
        "Sim", "Sim", "Sim", "Não", "Sim",
        "Sim", "Sim", "Não", "Sim", "Sim"
    ],
    "orientacao_proximo_passo": [
        "Sim", "Não", "Sim", "Não", "Não",
        "Sim", "Não", "Sim", "Não", "Sim",
        "Não", "Sim", "Não", "Sim", "Não"
    ]
}

df = pd.DataFrame(dados)

df

,id_solicitacao,idade,motivo_negativa,motivo_informado,orientacao_proximo_passo
0,1,22,Renda insuficiente,Sim,Sim
1,2,25,Motivo não informado,Não,Não
2,3,34,Documentação incompleta,Sim,Sim
3,4,41,Renda insuficiente,Sim,Não
4,5,28,Motivo não informado,Não,Não
5,6,52,Critérios não atendidos,Sim,Sim
6,7,37,Documentação incompleta,Sim,Não
7,8,24,Renda insuficiente,Sim,Sim
8,9,46,Motivo não informado,Não,Não
9,10,33,Critérios não atendidos,Sim,Sim


In [3]:
df["motivo_negativa"].value_counts()

,count
motivo_negativa,
Renda insuficiente,5
Motivo não informado,4
Documentação incompleta,3
Critérios não atendidos,3


In [4]:
(df["motivo_negativa"].value_counts(normalize=True) * 100).round(1)

,proportion
motivo_negativa,
Renda insuficiente,33.3
Motivo não informado,26.7
Documentação incompleta,20.0
Critérios não atendidos,20.0


In [7]:
df["faixa_etaria"] = pd.cut(
    df["idade"],
    bins=[0, 29, 44, 100],
    labels=["18-29", "30-44", "45+"]
)

analise_idade = df.groupby("faixa_etaria", observed=True).agg(
    total=("id_solicitacao", "count"),
    sem_orientacao=("orientacao_proximo_passo",
                    lambda x: (x == "Não").sum())
)

analise_idade["percentual_sem_orientacao"] = (
    analise_idade["sem_orientacao"] /
    analise_idade["total"] * 100
).round(1)

analise_idade

,total,sem_orientacao,percentual_sem_orientacao
faixa_etaria,,,
18-29,6,3,50.0
30-44,5,3,60.0
45+,4,2,50.0


Possíveis causas das negativas financeiras

Foram levantadas quatro hipóteses que podem estar relacionadas às negativas:

1.Cadastro desatualizado.

2.Renda incompatível com a solicitação.

3.Critérios de idade.

4.Risco de crédito.

Esses fatores serão considerados pontos de investigação e precisam ser validados com dados reais.

In [10]:
orientacoes = {
    "Renda insuficiente": "Verifique e atualize seus dados de renda.",
    "Motivo não informado": "Entre em contato para esclarecer a negativa.",
    "Documentação incompleta": "Confira os documentos pendentes.",
    "Critérios não atendidos": "Consulte os critérios necessários."
}

df["solucao_proposta"] = df["motivo_negativa"].map(orientacoes)

df[["motivo_negativa", "solucao_proposta"]]

,motivo_negativa,solucao_proposta
0,Renda insuficiente,Verifique e atualize seus dados de renda.
1,Motivo não informado,Entre em contato para esclarecer a negativa.
2,Documentação incompleta,Confira os documentos pendentes.
3,Renda insuficiente,Verifique e atualize seus dados de renda.
4,Motivo não informado,Entre em contato para esclarecer a negativa.
5,Critérios não atendidos,Consulte os critérios necessários.
6,Documentação incompleta,Confira os documentos pendentes.
7,Renda insuficiente,Verifique e atualize seus dados de renda.
8,Motivo não informado,Entre em contato para esclarecer a negativa.
9,Critérios não atendidos,Consulte os critérios necessários.


In [12]:
pd.crosstab(
    df["faixa_etaria"],
    df["motivo_negativa"]
)

motivo_negativa,Critérios não atendidos,Documentação incompleta,Motivo não informado,Renda insuficiente
faixa_etaria,,,,
18-29,1,0,2,3
30-44,1,2,1,1
45+,1,1,1,1


In [13]:
(pd.crosstab(
    df["faixa_etaria"],
    df["motivo_negativa"],
    normalize="index"
) * 100).round(1)

motivo_negativa,Critérios não atendidos,Documentação incompleta,Motivo não informado,Renda insuficiente
faixa_etaria,,,,
18-29,16.7,0.0,33.3,50.0
30-44,20.0,40.0,20.0,20.0
45+,25.0,25.0,25.0,25.0


## Resultado do cruzamento por faixa etária

A análise identificou diferenças nos motivos das negativas entre as faixas etárias.

- 18 a 29 anos: renda insuficiente (50%).
- 30 a 44 anos: documentação incompleta (40%).
- 45 anos ou mais: distribuição equilibrada entre os motivos (25% cada).

A faixa de 30 a 44 anos também apresentou o maior percentual de solicitações sem orientação (60%), indicando uma oportunidade de melhoria na comunicação com esse público.

## Solução proposta

Propor mensagens padronizadas para informar o motivo da negativa e orientar o cliente sobre os próximos passos.

As orientações serão personalizadas conforme o motivo, incluindo atualização cadastral, revisão de renda e envio de documentos pendentes.

O objetivo é reduzir dúvidas, facilitar a compreensão e melhorar a experiência do cliente.

## Conclusão

O projeto permitiu analisar os motivos das negativas financeiras e identificar oportunidades de melhoria na comunicação com os clientes.

A análise destacou a faixa de 30 a 44 anos como o público com maior percentual de solicitações sem orientação (60%).

Como solução, foi proposta a utilização de mensagens padronizadas, com orientações específicas para cada motivo de negativa.

Com isso, buscamos contribuir para uma comunicação mais clara e uma melhor experiência do cliente.